# Week 8 Demo  
## Predicting Energy Demand Using the UCI Appliance Energy Dataset

In this demo, we use the **UCI Appliance Energy Prediction dataset** to predict appliance energy use in a low-energy house.

The target variable is:

```text
Appliances
```

This is a **supervised learning regression** problem because we use input features to predict a numerical energy-use value.

## What this notebook does

1. Downloads the dataset from UCI.
2. Saves a cleaned CSV file that can also be used in Orange.
3. Selects a beginner-friendly set of features.
4. Trains two models:
   - Linear Regression
   - Random Forest Regression
5. Compares the models using:
   - MAE
   - RMSE
   - R²
6. Plots observed vs predicted appliance energy use.
7. Shows feature importance for the Random Forest model.

## Step 1: Install and import packages

In [ ]:
# Run this cell first.
# The ucimlrepo package lets us download datasets directly from the UCI Machine Learning Repository.

!pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 2: Download the UCI Appliance Energy dataset

The UCI dataset ID for **Appliances Energy Prediction** is `374`.

This cell downloads the dataset and combines the input features and target into one table.

In [ ]:
appliance_data = fetch_ucirepo(id=374)

X = appliance_data.data.features
y = appliance_data.data.targets

df = pd.concat([X, y], axis=1)

print("Rows and columns:", df.shape)
df.head()

## Step 3: Save a cleaned CSV for Orange

This CSV can be downloaded and opened in Orange using the **File** widget.

We keep the original useful columns, but later we will choose a smaller feature set for the model.

In [ ]:
clean_csv_name = "Wk8_UCI_Appliance_Energy_Cleaned.csv"

df.to_csv(clean_csv_name, index=False)

print("Saved:", clean_csv_name)
print("You can download this file from the Colab file panel, or run the next cell.")

### Optional: download the cleaned CSV to your computer

In [ ]:
from google.colab import files

files.download(clean_csv_name)

## Step 4: Inspect the dataset

The target variable is:

```text
Appliances
```

This represents appliance energy use in Wh.

The dataset includes indoor temperature, indoor humidity, weather, lighting, and time-related information.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head())

print("\nTarget summary:")
print(df["Appliances"].describe())

## Step 5: Create simple time features

The original dataset has a `date` column. Machine learning models cannot directly use a date/time string, so we create simple time features:

- hour of day
- day of week

These features help represent daily and weekly patterns in appliance energy use.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek

df[["date", "hour", "day_of_week", "Appliances"]].head()

## Step 6: Choose features and target

For a beginner-friendly model, we use a smaller set of meaningful features.

You can later experiment by adding or removing features.

In [ ]:
target = "Appliances"

features = [
    "lights",
    "T1", "RH_1",
    "T2", "RH_2",
    "T3", "RH_3",
    "T_out",
    "Press_mm_hg",
    "RH_out",
    "Windspeed",
    "hour",
    "day_of_week"
]

X = df[features]
y = df[target]

print("Features used:")
print(features)

print("\nTarget:")
print(target)

## Step 7: Split into training and testing data

We use a 70/30 split.

The model learns from the training data and is evaluated on the testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## Step 8: Train two regression models

We compare:

1. **Linear Regression** — a simple baseline model.
2. **Random Forest Regression** — a more flexible machine learning model.

The goal is not to build a perfect energy management system. The goal is to compare how different models learn patterns from the same data.

In [ ]:
linear_model = LinearRegression()

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

linear_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Models trained.")

## Step 9: Evaluate the models

We use three common regression metrics:

- **MAE**: average absolute error
- **RMSE**: larger errors are penalized more strongly
- **R²**: how much variation the model explains

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    }, predictions

linear_results, linear_predictions = evaluate_model(
    linear_model, X_test, y_test, "Linear Regression"
)

rf_results, rf_predictions = evaluate_model(
    rf_model, X_test, y_test, "Random Forest Regression"
)

results_df = pd.DataFrame([linear_results, rf_results])
results_df

## Step 10: Plot observed vs predicted values

A perfect model would place points close to the diagonal line.

In real environmental and energy data, predictions are usually imperfect.

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(y_test, linear_predictions, alpha=0.3, label="Linear Regression")
plt.scatter(y_test, rf_predictions, alpha=0.3, label="Random Forest Regression")

min_val = min(y_test.min(), linear_predictions.min(), rf_predictions.min())
max_val = max(y_test.max(), linear_predictions.max(), rf_predictions.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1)

plt.xlabel("Observed appliance energy use (Wh)")
plt.ylabel("Predicted appliance energy use (Wh)")
plt.title("Observed vs Predicted Appliance Energy Use")
plt.legend()
plt.show()

## Step 11: Feature importance from Random Forest

Feature importance gives a simple way to see which variables helped the Random Forest model make predictions.

This is useful for interpretation, but it should not be treated as a perfect explanation.

In [ ]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.show()

## Step 12: Try your own feature set

Use this optional cell to experiment.

For example, try removing some variables or adding more temperature/humidity variables.

In [ ]:
# Example: try a smaller feature set
features_try = [
    "lights",
    "T1", "RH_1",
    "T_out",
    "RH_out",
    "hour"
]

X_try = df[features_try]
y_try = df[target]

X_train_try, X_test_try, y_train_try, y_test_try = train_test_split(
    X_try, y_try,
    test_size=0.30,
    random_state=42
)

model_try = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model_try.fit(X_train_try, y_train_try)
pred_try = model_try.predict(X_test_try)

print("Feature set tested:", features_try)
print("MAE:", round(mean_absolute_error(y_test_try, pred_try), 2))
print("RMSE:", round(mean_squared_error(y_test_try, pred_try) ** 0.5, 2))
print("R²:", round(r2_score(y_test_try, pred_try), 3))

# Orange workflow

Use the cleaned CSV created by this notebook:

```text
Wk8_UCI_Appliance_Energy_Cleaned.csv
```

Suggested Orange workflow:

```text
File → Select Columns → Test and Score
```

Connect these models to **Test and Score**:

```text
Linear Regression
Random Forest
```

In **Select Columns**:

- Target: `Appliances`
- Features: selected temperature, humidity, weather, lighting, and time variables
- Meta: `date`

Then connect:

```text
Test and Score → Predictions → Data Table
```

Orange users may not need to create `hour` and `day_of_week` unless the instructor provides a prepared CSV with those columns.

# Reflection questions

Answer these in your assignment report.

1. What is the target variable?
2. What input features did you use?
3. Why is this a supervised learning problem?
4. Why is this a regression problem?
5. Which model performed better: Linear Regression or Random Forest Regression?
6. Which model had lower MAE and RMSE?
7. Which model had higher R²?
8. Which features seemed useful for prediction?
9. How could this model support energy management?
10. What are the limitations of using this model for real energy decisions?